# ATINA — gerar imagens das soluções com IA (Kaggle)

Notebook preparado para executar o gerador do ATINA numa GPU Kaggle, quando disponível. Mantém este notebook **Private**. O CSV contém respostas e pistas: adiciona-o apenas como input privado. Não publiques o dataset nem o notebook.

A GPU gratuita tem quota e disponibilidade limitadas. Começa com 5 imagens. O envio para Supabase é opcional e usa Kaggle Secrets.

## 1. Ativar GPU e Internet

No painel Settings / Session options escolhe **Accelerator: GPU** (P100 ou outra opção disponível) e ativa **Internet**. A GPU pode estar em fila ou indisponível.

In [ ]:
import torch, os
print('CUDA disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Ativa uma GPU nas definições do Kaggle antes de continuar.')


## 2. Adicionar o CSV privado

No painel **Input / Add Input**, usa **Upload** para carregar o CSV exportado do Supabase, ou adiciona o teu dataset Kaggle privado. Não publiques o CSV. A célula seguinte procura automaticamente o ficheiro CSV em `/kaggle/input`.

In [ ]:
from pathlib import Path
csv_files = list(Path('/kaggle/input').rglob('*.csv'))
print('CSV encontrados:')
for i, p in enumerate(csv_files): print(i, p)
assert csv_files, 'Não encontrei CSV. Adiciona o ficheiro em Input > Add Input > Upload.'
matches = [p for p in csv_files if 'challenge' in p.name.lower()]
CSV_PATH = str(matches[0] if matches else csv_files[0])
print('Manifesto selecionado:', Path(CSV_PATH).name)


## 3. Instalar o gerador ATINA

O código vem diretamente da branch de preparação do projeto. A instalação e a primeira descarga do modelo podem demorar.

In [ ]:
!git clone --depth 1 --branch feature/ai-solution-images https://github.com/lufime/ATINA.git /kaggle/working/ATINA
%cd /kaggle/working/ATINA/tools/ai-solution-images
!pip -q install -r requirements.txt


## 4. Configurar credenciais privadas do Supabase (opcional)

Para enviar imagens diretamente para o bucket ATINA, abre **Add-ons → Secrets** e cria `SUPABASE_URL` e `SUPABASE_SERVICE_ROLE_KEY`. Ativa o acesso destes secrets para este notebook. Nunca escrevas a chave numa célula, output, CSV ou GitHub. Se ainda não quiseres configurar, podes gerar localmente no armazenamento temporário e descarregar os ficheiros depois.

In [ ]:
from kaggle_secrets import UserSecretsClient
from getpass import getpass
UPLOAD_TO_SUPABASE = False
try:
    secrets = UserSecretsClient()
    os.environ['SUPABASE_URL'] = secrets.get_secret('SUPABASE_URL')
    os.environ['SUPABASE_SERVICE_ROLE_KEY'] = secrets.get_secret('SUPABASE_SERVICE_ROLE_KEY')
    UPLOAD_TO_SUPABASE = bool(os.environ['SUPABASE_URL'] and os.environ['SUPABASE_SERVICE_ROLE_KEY'])
except Exception:
    print('Secrets não configurados. As imagens serão geradas sem upload.')
print('Upload Supabase ativo:', UPLOAD_TO_SUPABASE)


## 5. Testar com 5 imagens

Executa primeiro esta célula. Se o envio estiver configurado, os ficheiros são enviados para Storage e o SQL é criado para revisão. O script marca os ativos como `generated`, nunca como aprovados.

In [ ]:
import subprocess
cmd = ['python', 'generate.py', '--manifest', CSV_PATH, '--output', '/kaggle/working/ATINA-generated/images', '--limit', '5']
if UPLOAD_TO_SUPABASE: cmd.append('--upload')
subprocess.run(cmd, check=False)


## 6. Gerar o conjunto completo

Só executa depois de verificares as primeiras imagens. A geração das 1.153 pode exigir várias sessões e a quota gratuita pode não chegar para tudo. Se interromper, guarda a versão/output do notebook e retoma os lotes em sessões seguintes. As imagens só são consideradas integradas após revisão humana e aplicação do SQL.

In [ ]:
cmd = ['python', 'generate.py', '--manifest', CSV_PATH, '--output', '/kaggle/working/ATINA-generated/images']
if UPLOAD_TO_SUPABASE: cmd.append('--upload')
subprocess.run(cmd, check=False)


## 7. Rever e guardar o SQL

O SQL gerado está em `/kaggle/working/ATINA/tools/ai-solution-images/generated/apply-generated-images.sql`. Revê as imagens e o SQL antes de o executar no Supabase SQL Editor. Não publiques o SQL: contém URLs associadas às respostas dos desafios.

In [ ]:
from pathlib import Path
sql_path = Path('/kaggle/working/ATINA/tools/ai-solution-images/generated/apply-generated-images.sql')
print('SQL criado:', sql_path.exists())
if sql_path.exists(): print(sql_path)
print('Imagens geradas:', len(list(Path('/kaggle/working/ATINA-generated/images').glob('*.webp'))))
